# Lecture 06 companion notebook — Python

**BIOE 610 Computational Tools · Week 6**

Every example from Lecture 06, plus this week's self-study, as cells you can run, change and break — in lecture order: files → data frames → random draws → pandas chaining → seaborn → the figure gallery, with reference sections on control flow, gotchas and error messages at the end. There is a matching R notebook (`Lecture06_R_companion.ipynb`) with the same section numbers — keep them open side by side to compare languages.

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick a **Python interpreter** if prompted (top right)
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

## 1 · The packages this notebook uses

`pip install <package>` puts a package in the *active environment* — **once** per environment. `import` makes it available in *this* session — **every** session (`ModuleNotFoundError` means it is not installed *in this environment*; `which python3` tells you which one that is). Run this first:

In [ ]:
import numpy as np                 # arrays and random generators
import pandas as pd                # data frames
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # statistical graphics
from pathlib import Path           # file paths
from scipy.stats import norm       # theoretical densities (section 9)
# section 29 (PCA) additionally needs: pip install scikit-learn

# Part 1 · Reading and writing files

## 2 · Reading tabular data

`pd.read_csv()` handles comma- *and* tab-separated files — `sep="\t"` makes it tab-separated — and reads straight from a URL, so this notebook works wherever you saved it.

In [ ]:
stickle = pd.read_csv(
    "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
    sep="\t")
print(stickle.shape)          # (rows, columns)
stickle.iloc[0:3, 0:5]        # first rows, first columns

## 3 · Writing tabular data

Write **derived** tables into `results/` — never overwrite anything in `data/` (raw data is read-only). `index=False`, or pandas sneaks an extra unnamed column of row numbers into your file.

In [ ]:
genes  = [c for c in stickle.columns if c.startswith("Gene")]
totals = pd.DataFrame({"Individual": stickle["Individual"],
                       "Total":      stickle[genes].sum(axis=1)})
totals.head(3)

In [ ]:
Path("results").mkdir(exist_ok=True)
totals.to_csv("results/totals.csv", index=False)
totals.to_csv("results/totals.tsv", sep="\t", index=False)
sorted(p.name for p in Path("results").iterdir())

## 4 · Saving figures to files

matplotlib's pattern is simply `plt.savefig()` — the extension picks the format. Vector formats (`.svg`, `.pdf`) scale forever; raster (`.png`) needs a resolution (`dpi=300`).

In [ ]:
plt.boxplot(totals["Total"])
plt.ylabel("Total reads per fish")
plt.savefig("results/first_figure.svg")
plt.savefig("results/first_figure.png", dpi=300)
plt.show()

sorted(p.name for p in Path("results").iterdir())

# Part 2 · Data frames

## 5 · Creating a data frame by hand

A `DataFrame` is built from a dictionary of `column name: values`. The row names are the **index**.

In [ ]:
hydrogel_concentration = pd.Categorical(["low", "high", "high", "high",
                                         "medium", "medium", "medium", "low"])
compression  = [3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5]
conductivity = [0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3]

mydata = pd.DataFrame({"hydrogel_concentration": hydrogel_concentration,
                       "compression": compression,
                       "conductivity": conductivity})
mydata.index = [f"Sample_{i}" for i in range(1, 9)]
mydata

## 6 · The stickleback table as a data frame

First questions for any new table: how big (`.shape`), what's in it (`.head()`, `.info()`), how do the groups break down (`pd.crosstab`), what does one column look like (`.describe()`).

In [ ]:
print(stickle.shape)
pd.crosstab(stickle["Genotype"], stickle["Microbiota"])

In [ ]:
stickle["Gene1"].describe()

## 7 · Summary statistics by group

`.groupby()` + an aggregation is the pandas counterpart of R's `aggregate()`; it generalizes to many statistics at once with `.agg()` (the R counterpart is `group_by() |> summarize()`, section 19 of the R notebook).

In [ ]:
stickle.groupby(["Genotype", "Microbiota"])["Gene1"].median()

In [ ]:
stickle[["Gene1", "Gene2", "Gene3"]].mean()

# Part 3 · Random draws from known distributions

### Why draw random numbers? And with which arguments?

A scientist generates data constantly: to **test an analysis** on data where the right answer is known, to **build a null model** (what would the data look like if nothing were going on?), and to **plan experiments** by simulation. Each distribution is a different data-generating machine, with its own parameters as arguments — and the first argument is always `n`, the number of draws:

| Distribution | R | Python (numpy generator) |
|:---|:---|:---|
| Uniform | `runif(n, min = 0, max = 100)` | `rng.uniform(0, 100, n)` |
| Normal | `rnorm(n, mean = 50, sd = 10)` | `rng.normal(50, 10, n)` |
| Poisson | `rpois(n, lambda = 5)` | `rng.poisson(5, n)` |

`set.seed()` (R) / `np.random.default_rng(seed)` (Python) make the draws reproducible: same seed, same "random" numbers.

## 8 · Drawing random samples

In Python random numbers come from a numpy **generator**: `rng = np.random.default_rng(2026)` — the number is the seed, like `set.seed(2026)` in R. The generator's methods (`normal`, `uniform`, `poisson`, `binomial`, `integers`, `choice`) are the `r*` family.

In [ ]:
rng = np.random.default_rng(2026)
x = rng.normal(0, 10, 10000)                 # rnorm(10000, 0, 10)
y = rng.integers(1, 10001, size=10000)       # sample(1:10000, ..., replace = TRUE)
plt.scatter(x, y, s=2, alpha=0.4, color="#31688e")
plt.xlabel("x : rng.normal(0, 10)"); plt.ylabel("y : rng.integers(1, 10001)")
plt.show()

In [ ]:
# Your turn within a cell: change the mean and sd and re-run.
# What does sd = 1 vs sd = 50 do to the histogram?
plt.hist(rng.normal(0, 10, 10000), bins=40)
plt.show()

## 9 · Histograms and the honest density overlay

`density=True` puts the bars on the **density** scale, so the theoretical curve `norm.pdf()` drops straight on top — no magic scaling constants. Every `plt.` call draws on the same figure until `plt.show()`.

In [ ]:
from scipy.stats import norm

x = rng.normal(0, 100, 1000)
plt.hist(x, bins=30, density=True, color="lightgrey", edgecolor="white")
grid = np.linspace(-400, 400, 200)
plt.plot(grid, norm.pdf(grid, 0, 100), color="#440154", lw=2)
plt.xlabel("x"); plt.ylabel("density")
plt.show()

## 10 · Three distributions at once — and multi-panel figures

`plt.subplots(1, 3)` hands you the grid as an array of **axes** to draw on. Three workhorses, as in the lecture: **uniform** (anything equally likely), **normal** (measurements), **Poisson** (counts). In a script you would add `plt.savefig("results/distributions.svg")` before `plt.show()` to save it.

In [ ]:
n = 2000
draws = {"Uniform(0, 100)": (rng.uniform(0, 100, n), "#31688e"),
         "Normal(50, 10)":  (rng.normal(50, 10, n),  "#440154"),
         "Poisson(5)":      (rng.poisson(5, n),      "#35b779")}

fig, ax = plt.subplots(1, 3, figsize=(10, 3.6))    # the grid, as an array of axes
for a, (name, (d, c)) in zip(ax, draws.items()):
    a.hist(d, bins=40, color=c, edgecolor="white")
    a.set_title(name)
fig.tight_layout()
plt.show()

In [ ]:
# Change n, the parameters and bins until each panel surprises you.
# What happens to Poisson(5) as you raise lam to 50?
# Then add a fourth panel: rng.binomial(20, 0.3, n) - successes out of 20 tries.
plt.hist(rng.poisson(50, 2000), bins=40, color="#35b779", edgecolor="white")
plt.show()

# Part 4 · matplotlib vocabulary and first figures (self-study)

## 11 · matplotlib vocabulary

matplotlib layers the same way base R does: a first `plt.plot()` starts the figure, further calls add to it, and `plt.xlabel()` / `plt.title()` / `plt.legend()` do the styling.

In [ ]:
seq_1 = np.linspace(0, 10, 101)
plt.plot(seq_1, np.sin(seq_1), lw=2, color="#440154")             # the line
plt.plot(seq_1[::10], np.sin(seq_1)[::10], "o", color="#35b779")  # every 10th point
plt.xlabel("x"); plt.ylabel("sin(x)")
plt.title("matplotlib layers the same way")
plt.legend(["line", "every 10th point"])
plt.show()

## 12 · A first real figure

In [ ]:
stickle.boxplot(column="Gene1", by=["Genotype", "Microbiota"])
plt.suptitle("")
plt.title("Gene1 counts by host genotype and microbiota")
plt.ylabel("Gene1 read count")
plt.show()

## 13 · The same figure, dressed for a lab meeting

seaborn derives the grouping, colors and legend *from the data* — sections 21-23 take that grammar-of-graphics idea the rest of the way.

In [ ]:
import seaborn as sns

sns.boxplot(data=stickle, x="Genotype", y="Gene1", hue="Microbiota",
            palette=["#440154", "#35b779"])
sns.stripplot(data=stickle, x="Genotype", y="Gene1", hue="Microbiota",
              dodge=True, size=4, alpha=0.6, palette="dark:black", legend=False)
plt.xlabel("Host genotype"); plt.ylabel("Gene1 read count")
plt.title("Stickleback gut RNA-seq: Gene1 by group")
plt.show()

## 14 · Inspecting objects

When an object surprises you, interrogate it first. These five answer most questions.

In [ ]:
stickle = pd.read_csv(
    "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
    sep="\t")

type(stickle)
print(stickle.shape)
list(stickle.columns[:6])

In [ ]:
stickle.iloc[:, :5].info()        # columns, dtypes, non-null counts
stickle["Gene1"].describe()       # quick summary of one column

## 15 · Logical operators and membership

Comparisons return boolean arrays; boolean arrays select rows. New today: **membership** with `np.isin()` / `.isin()`. In pandas/numpy use `&`, `|`, `~` with **parentheses** around each comparison.

In [ ]:
x = np.array([1, 5, 10, 15, 20])
x > 10                      # one True/False per element
x[x > 10]                   # keep only where True - L05's boolean indexing
np.isin(x, [1, 15, 99])     # membership: is each element in the set?

## 16 · Control flow: making decisions with `if`

Python's blocks are the **indentation** (4 spaces), and `else if` becomes `elif`. The vectorized cousin `np.where()` gives one answer per element — no loop needed.

In [ ]:
mass = 1.42
if mass > 1.0:
    print("big fish")          # the indented lines ARE the block
elif mass > 0.5:
    print("medium fish")
else:
    print("small fish")

In [ ]:
x = np.array([0.3, 0.8, 1.42])
np.where(x > 1.0, "big", "small")    # vectorized: one answer per element

## 17 · Control flow: repeating with `for` (and `while`)

Both languages loop directly over a collection's *elements* — no counters required. `enumerate()` gives position **and** element (0-based!). `break` leaves early; `continue` skips to the next turn.

In [ ]:
genes = ["tp53", "brca1", "myc"]
for g in genes:
    print(g.upper())

for i, g in enumerate(genes):     # position AND element (0-based!)
    print(i, g)

In [ ]:
i = 1
while i < 8:                      # repeat until the condition fails
    i = i * 2
print(i)

## 18 · When you do *not* need a loop

Vectorized code is shorter, faster and easier to read. Loops earn their keep for files, simulations, and step-*n*-depends-on-step-*n−1* problems.

In [ ]:
x = np.array([2, 3, 4, 5])
print(x**2)                                   # not: for i in range(len(x)) ...
genes_cols = [c for c in stickle.columns if c.startswith("Gene")]
stickle[genes_cols].sum(axis=1).head()        # not: nested loops

## 19 · Method chaining: pandas' pipe

Each method takes a DataFrame and returns a DataFrame, so they chain with the dot. Wrap the chain in `( )` to put each step on its own line.

In [ ]:
mydata = pd.DataFrame({
    "hydrogel_concentration": pd.Categorical(["low", "high", "high", "high",
                                              "medium", "medium", "medium", "low"]),
    "compression":  [3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5],
    "conductivity": [0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3]})

(mydata
   .query("compression > 4")                                  # keep rows
   .assign(ratio=lambda d: d.conductivity / d.compression)    # add a column
   .groupby("hydrogel_concentration", observed=True)          # define groups
   .agg(n=("ratio", "size"), mean_ratio=("ratio", "mean")))   # one row per group

## 20 · Wide to long: `.melt()`

One `.melt()` unlocks *every* gene at once: 80 × 600 = 48,000 rows, one per fish × gene.

In [ ]:
stickle_long = stickle.melt(
    id_vars=["Individual", "Genotype", "Microbiota", "Geno&Micro"],
    var_name="gene", value_name="count")
print(stickle_long.shape)
stickle_long.head(3)

In [ ]:
gene_means = (stickle_long
    .groupby(["gene", "Genotype", "Microbiota"], as_index=False)
    .agg(mean_count=("count", "mean")))
gene_means.query("gene == 'Gene1'")

## 21 · seaborn, layer 1: data + mappings + a plot kind

`sns.stripplot()` is a jittered scatter — every observation visible, none overprinting.

In [ ]:
gene1 = stickle_long.query("gene == 'Gene1'")
sns.stripplot(data=gene1, x="Genotype", y="count", jitter=0.08)
plt.show()

## 22 · Layer 2: more mappings, more layers

Mapping a column to `hue` splits the groups **and** builds the legend — automatically, from the data. Later calls draw on the same axes.

In [ ]:
sns.violinplot(data=gene1, x="Genotype", y="count", hue="Microbiota",
               inner=None, alpha=0.4, palette=["#440154", "#35b779"])
sns.stripplot(data=gene1, x="Genotype", y="count", hue="Microbiota",
              dodge=True, size=4, alpha=0.6, legend=False,
              palette="dark:black")
plt.show()

## 23 · Layers 3–5: facets, colors, labels — and `savefig()`

`col="gene"` in `sns.catplot()` gives one panel per gene for free. This is the left half of L06's destination figure.

In [ ]:
long4 = stickle_long[stickle_long["gene"].isin([f"Gene{i}" for i in range(1, 5)])]

g = sns.catplot(data=long4, x="Genotype", y="count", hue="Microbiota",
                col="gene", col_wrap=2, kind="violin", inner="point",
                palette=["#440154", "#35b779"], sharey=False, height=2.6, aspect=1.3)
g.set_axis_labels("Host genotype", "Read count")
g.set_titles("{col_name}")
g.figure.suptitle("Stickleback gut RNA-seq", y=1.02)
plt.show()

In [ ]:
Path("results").mkdir(exist_ok=True)
g.figure.savefig("results/gene_panel.svg", bbox_inches="tight")
sorted(p.name for p in Path("results").iterdir())

## 24 · Six geoms, one example each

The lecture's geom tour as runnable cells — histogram, boxplot, scatter, trend, line and bars, all from the stickleback table. Each ends with the save-line convention: `plt.savefig()` into `results/`.

In [ ]:
# sns.histplot(): one continuous column; the function does the binning
sns.histplot(data=gene1, x="count", bins=30, color="#31688e")
plt.xlabel("Gene1 read count"); plt.ylabel("Number of fish")
plt.savefig("results/gene1_histogram.svg")
plt.show()

In [ ]:
# sns.boxplot(): continuous by category; a second category dodges via hue
gene2 = stickle_long.query("gene == 'Gene2'")
sns.boxplot(data=gene2, x="Genotype", y="count", hue="Microbiota",
            palette=["#440154", "#35b779"])
plt.xlabel("Host genotype"); plt.ylabel("Gene2 read count")
plt.savefig("results/gene2_boxplot.svg")
plt.show()

In [ ]:
# sns.scatterplot(): two continuous columns; a third rides along as hue
sns.scatterplot(data=stickle, x="Gene1", y="Gene2", hue="Genotype",
                s=50, alpha=0.7)
plt.savefig("results/gene1_gene2_scatter.svg")
plt.show()

In [ ]:
# sns.regplot(): a fitted trend layered on the scatter
sns.regplot(data=stickle, x="Gene1", y="Gene2",
            scatter_kws={"alpha": 0.5, "color": "#31688e"},
            line_kws={"color": "#440154"})
plt.savefig("results/gene1_gene2_trend.svg")
plt.show()

In [ ]:
# ax.plot(): lines need an ORDERED x - here, genes ranked by mean expression
genes = [c for c in stickle.columns if c.startswith("Gene")]
means = stickle[genes].mean().sort_values(ascending=False)

fig, ax = plt.subplots()
ax.plot(range(1, len(means) + 1), means, color="#440154", lw=2)
ax.set(yscale="log", xlabel="Gene rank (by mean expression)",
       ylabel="Mean read count (log)")
plt.savefig("results/gene_rank_means.svg")
plt.show()

In [ ]:
# sns.barplot(): bars of a statistic you choose (here the median)
sns.barplot(data=gene1, x="Genotype", y="count", hue="Microbiota",
            estimator=np.median, errorbar=None,
            palette=["#440154", "#35b779"])
plt.xlabel("Host genotype"); plt.ylabel("Median Gene1 read count")
plt.savefig("results/gene1_group_medians.svg")
plt.show()

## 25 · The closer: all 600 genes in one figure

One mean and one variance per gene — 48,000 rows collapsed to 600 points. Dashed line: variance = mean (Poisson); real RNA-seq runs hotter.

In [ ]:
genes = [c for c in stickle.columns if c.startswith("Gene")]
mv = pd.DataFrame({"mean": stickle[genes].mean(), "var": stickle[genes].var()})

fig, ax = plt.subplots()
ax.scatter(mv["mean"], mv["var"], s=10, alpha=0.4, color="#31688e")
lims = np.array([mv["mean"].min() / 2, mv["mean"].max() * 2])
ax.plot(lims, lims, ls="--", color="black")              # variance = mean
ax.set(xscale="log", yscale="log",
       xlabel="Mean count per gene (log)", ylabel="Variance (log)",
       title="All 600 genes at once")
plt.show()

# The figure gallery

## 26 · A clustered heatmap, from synthesized expression data

`z_score=0` standardizes each gene, so the colors compare *patterns*, not levels. The dendrograms should rediscover the two modules and the control/treated split we build in.

In [ ]:
rng = np.random.default_rng(2026)
expr = pd.DataFrame(
    rng.normal(8, 0.4, size=(30, 12)) + rng.normal(0, 1, size=(30, 1)),
    index=[f"gene_{i+1}" for i in range(30)],
    columns=[f"ctrl_{i+1}" for i in range(6)] + [f"treat_{i+1}" for i in range(6)])
expr.iloc[:15, 6:] += 2.0          # module 1: up in treated
expr.iloc[15:, 6:] -= 1.5          # module 2: down in treated

sns.clustermap(expr, z_score=0, cmap="viridis", figsize=(7, 6))
plt.show()

## 27 · Growth curves with ribbons

`plt.fill_between()` draws the ribbon. The pattern works for any repeated time course: sensor logs, qPCR, cyclic loading.

In [ ]:
t = np.arange(0, 49, 2)
strains = {"wild type": (1.8, 0.25), "mutant A": (1.2, 0.35), "mutant B": (1.6, 0.15)}

fig, ax = plt.subplots()
for (name, (K, r)), c in zip(strains.items(), ["#440154", "#31688e", "#35b779"]):
    reps = np.array([K / (1 + np.exp(-r * (t - 20))) + rng.normal(0, 0.05, len(t))
                     for _ in range(4)])
    m, s = reps.mean(axis=0), reps.std(axis=0)
    ax.plot(t, m, color=c, lw=2, label=name)
    ax.fill_between(t, m - s, m + s, color=c, alpha=0.25)
ax.set(xlabel="Time (h)", ylabel="OD600",
       title="Bioreactor growth curves (mean ± SD of 4 replicates)")
ax.legend()
plt.show()

## 28 · Dose-response curves

The log-scaled x axis is what makes dose-response curves readable — doses span five orders of magnitude. The IC50 is where each curve crosses 50%.

In [ ]:
dose = np.logspace(-9, -4, 8)
fig, ax = plt.subplots()
for (name, ic50), c in zip({"compound X": 3e-7, "compound Y": 5e-6}.items(),
                           ["#440154", "#35b779"]):
    for _ in range(3):
        viab = 100 / (1 + (dose / ic50) ** 1.2) + rng.normal(0, 4, len(dose))
        ax.scatter(dose, viab, s=18, alpha=0.6, color=c)
    grid = np.logspace(-9, -4, 200)
    ax.plot(grid, 100 / (1 + (grid / ic50) ** 1.2), color=c, lw=2, label=name)
ax.set(xscale="log", xlabel="Dose (M)", ylabel="Viability (%)",
       title="Dose-response: two compounds, three replicates")
ax.legend()
plt.show()

## 29 · PCA of the real stickleback fish

PCA compresses 600 correlated gene columns into a few axes of maximal variation. `np.log1p()` tames the counts; standardizing weights every gene equally. Needs `pip install scikit-learn` (once).

In [ ]:
from sklearn.decomposition import PCA

X = np.log1p(stickle[genes].to_numpy())
X = (X - X.mean(axis=0)) / X.std(axis=0)          # like R's scale. = TRUE
pca = PCA(n_components=2).fit(X)
scores = pca.transform(X)
print("variance explained:", pca.explained_variance_ratio_.round(3))

In [ ]:
fig, ax = plt.subplots()
for grp, c in zip(sorted(stickle["Geno&Micro"].unique()),
                  ["#440154", "#31688e", "#35b779", "#fde725"]):
    sel = stickle["Geno&Micro"] == grp
    ax.scatter(scores[sel, 0], scores[sel, 1], s=30, alpha=0.8, color=c, label=grp)
ax.set(xlabel="PC1", ylabel="PC2", title="PCA of 80 fish from 600 genes")
ax.legend(title="Group")
plt.show()

## 30 · Common gotchas (reference)

Return here when something bites:

- **Python counts from 0**, `x[-1]` *is* the last element, and a slice `x[1:3]` **excludes** the end (elements 1 and 2)
- **`=` vs `==`**: `=` assigns, `==` tests equality
- **`^` is not power**: use `**`
- **`y = x` does not copy a list or array** - both names point to the same object; use `x.copy()`
- **`NaN` is contagious** in numpy, but pandas' `.mean()` skips it by default (`skipna=True`)
- **Floating point**: `0.1 + 0.2 == 0.3` is `False` - use `math.isclose()`
- **Indentation is syntax** - mixing tabs and spaces, or a missing `:`, gives `IndentationError` / `SyntaxError`
- **Wrong environment**: `ModuleNotFoundError` usually means the package is installed in a *different* Python - check `which python3` (environments get their own lecture in Week 9)
- **Working directory**: `FileNotFoundError` - check `Path.cwd()` and use project-relative paths
- **Whole-column conversion**: `pd.to_numeric(col, errors="coerce")` turns bad values into `NaN` instead of raising - then `col.isna()` finds them

## 31 · Reading error messages (reference)

| Error message | Meaning | Fix |
|:---|:---|:---|
| `NameError: name 'x' is not defined` | Variable doesn't exist | Check spelling; run the cell that creates it |
| `ModuleNotFoundError` | Package not installed *in this environment* | `pip install`, check `which python3` |
| `SyntaxError` / `IndentationError` | Missing `:`, bracket, or bad indentation | Look at the line **above** the arrow too |
| `TypeError: unsupported operand` | Wrong data type | Check with `type()` or `.dtypes` |
| `FileNotFoundError` | Wrong working directory or path | `Path.cwd()`, fix the path |

Python reports a **traceback** - the *last* line tells you what went wrong. When stuck, paste the *exact* message into a search engine or an AI assistant - then verify the answer against the documentation.

## Your turn

1. **Script the summaries**: write `scripts/summarize_stickleback.R` or `.py` that reads the data, writes a per-group median table for a gene of your choice to `results/`, and saves the distribution gallery as SVG *and* a 300-dpi PNG
2. **Rebuild Example 1** (the section-23 panel) for four *different* genes — in the language you know less well — and save it into `results/`
3. Color the mean-variance plot (section 25): compute each gene's mean separately for genotypes A and B, and color each point by which genotype is higher
4. **Stretch**: adapt the gallery heatmap to the real data — the 30 most *variable* stickleback genes, clustering the 80 fish (`scale = "row"` matters!)
5. Change the distribution gallery (section 10) until each panel surprises you — then add a fourth panel: the binomial

Working versions of the two scripts are on the Examples page — but only look after your own runs.